## Catastro Minero de Salta
### Notebook: Normalización de titularidad (`concesiona`)
### Proyecto: Análisis del catastro minero oficial de la provincia de Salta, Argentina
###Autora: Camila Mopty
#### Fecha: 2026

Objetivo: construir un identificador de titular normalizado y auditable para el Objetivo B, sin fusionar entidades distintas. El notebook genera *candidatos* de fusión; ninguna fusión se aplica sin aprobación manual.

## 1. Librerías y preparación

In [47]:
!pip install rapidfuzz -q

In [48]:
#@title
#LIBRERÍAS

import subprocess
import os
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import geopandas as gpd

from rapidfuzz import process, fuzz

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"
warnings.filterwarnings('ignore')

In [49]:
import importlib
for lib in ['numpy','pandas','geopandas','rapidfuzz','plotly']:
    try:
        print(f"{lib:12s} {importlib.import_module(lib).__version__}")
    except Exception as e:
        print(f"{lib:12s} no instalada ({e})")

numpy        2.1.3
pandas       2.2.3
geopandas    1.1.4
rapidfuzz    3.14.5
plotly       5.24.1


In [50]:
import os, random
import numpy as np

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [51]:
#@title
# Clonación del repositorio

repo_url = "https://github.com/Camilamop/catastro-minero-salta-mining-registry-salta.git"
repo_name = "catastro-minero-salta-mining-registry-salta"

if not os.path.exists(repo_name):
    subprocess.run(["git", "clone", repo_url], check=True)
else:
    print("El repositorio ya existe en el entorno.")

MODIFIED = Path(repo_name) / "data" / "modified"
MODIFIED.mkdir(parents=True, exist_ok=True)

El repositorio ya existe en el entorno.


## 2. Carga de los cortes

Se agrupan los titulares de los tres cortes para trabajar sobre el universo completo de nombres.

In [52]:
#@title
# Función de carga (misma lógica que el EDA)

CRITICOS = {'LI', 'CU', 'AU'}

def es_critico(valor):
    if pd.isna(valor):
        return False
    return bool({t.strip().upper() for t in str(valor).split(',')} & CRITICOS)

STEM = "poligonos_adaf55f39328ff45c8c60e94eb13a7a0.shp"

def cargar_corte(carpeta):
    g = gpd.read_file(f"{repo_name}/data/raw/{carpeta}/{STEM}", encoding='latin-1')
    g['area_ha'] = (
        g['area'].str.replace(' ha', '', regex=False).str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False).pipe(pd.to_numeric, errors='coerce')
    )
    g = g[g['mineral'].notna() & (g['mineral'].str.strip() != '')].copy()
    g['minerales_criticos'] = g['mineral'].apply(es_critico)
    return g

SNAPS = ['marzo_2026', 'junio_2026', 'agosto_2026']
titulares_raw = pd.concat(
    [cargar_corte(s)[['concesiona', 'area_ha']].assign(snapshot=s) for s in SNAPS],
    ignore_index=True
)
titulares_raw = titulares_raw[titulares_raw['concesiona'].notna() &
                              (titulares_raw['concesiona'].str.strip() != '')]
print("registros con titular:", len(titulares_raw))

registros con titular: 9383


## 3. Capa 1 — Normalización determinista

Reglas reproducibles: mayúsculas, sin acentos, formas jurídicas canonizadas y co-titulares ordenados alfabéticamente (para que 'A,B' y 'B,A' sean el mismo grupo). No fusiona entidades: solo estandariza la grafía.

Cada titular se clasifica en **empresa**, **persona física**, **mixto**, **gubernamental** o **cooperativa** según sus componentes.

In [53]:
#@title
# Normalización determinista y clasificación de tipo de actor

FORMAS = {r'\bS\s*\.?\s*A\s*\.?\s*U\b': 'SAU', r'\bSACIF\b': 'SACIF',
          r'\bS\s*\.?\s*A\s*\.?\s*S\b': 'SAS', r'\bS\s*\.?\s*A\b': 'SA',
          r'\bS\s*\.?\s*R\s*\.?\s*L\b': 'SRL', r'\bLIMITED\b|\bLTD\b': 'LTD',
          r'\bINC\b': 'INC', r'\bCORPORATION\b|\bCORP\b': 'CORP'}
FORMAS_SET = {'SAU', 'SACIF', 'SAS', 'SA', 'SRL', 'LTD', 'INC', 'CORP'}

def _token(s):
    s = ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))
    s = s.upper().replace('&', ' Y ')
    for p, r in FORMAS.items():
        s = re.sub(p, r, s)
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def normalizar_titular(s):
    if pd.isna(s):
        return s
    partes = sorted(p for p in (_token(x) for x in str(s).split(',')) if p)
    return ','.join(partes)

def tipo_actor(s):
    if pd.isna(s) or s == '':
        return 'sin_dato'
    subs = [x for x in str(s).split(',') if x]
    if any(p.startswith('MUNICIPALIDAD') or p.startswith('SECRETARIA') for p in subs):
        return 'gubernamental'
    if any('COMUNIDAD' in p for p in subs):
        return 'comunidad'
    if any('COOPERATIVA' in p for p in subs):
        return 'cooperativa'
    flags = [any(tok in FORMAS_SET for tok in sub.split()) for sub in subs]
    if all(flags):
        return 'empresa'
    if not any(flags):
        return 'persona_fisica'
    return 'mixto'

titulares_raw['titular_norm'] = titulares_raw['concesiona'].map(normalizar_titular)

titulares = (
    titulares_raw.groupby('titular_norm')
    .agg(n_registros=('titular_norm', 'size'),
         superficie_ha=('area_ha', 'sum'),
         ejemplo_original=('concesiona', 'first'))
    .reset_index()
)
titulares['tipo_actor'] = titulares['titular_norm'].map(tipo_actor)

print("titulares distintos (normalizados):", len(titulares))
display(titulares['tipo_actor'].value_counts().rename('cantidad').to_frame())

titulares distintos (normalizados): 772


,cantidad
tipo_actor,
persona_fisica,533
empresa,197
gubernamental,28
mixto,11
cooperativa,2
comunidad,1


In [54]:
#@title
# Gráfico 1: Distribución de titulares por tipo de actor

conteo = titulares['tipo_actor'].value_counts()
orden = ['empresa', 'persona_fisica', 'mixto', 'gubernamental', 'comunidad', 'cooperativa']
conteo = conteo.reindex([o for o in orden if o in conteo.index])
colores = {'empresa': '#C0392B', 'persona_fisica': '#4a5d7b', 'mixto': '#E8834A',
           'gubernamental': '#2C3E50', 'comunidad': '#27AE60', 'cooperativa': '#8E44AD'}

fig = go.Figure(go.Bar(
    x=conteo.index, y=conteo.values,
    marker_color=[colores[t] for t in conteo.index],
    text=conteo.values, textposition='outside'
))
fig.update_layout(
    separators='.,',
    title=dict(text='Titulares distintos por tipo de actor',
               font=dict(size=15, color='#2C3E50'), x=0.05),
    xaxis=dict(title=''),
    yaxis=dict(title='Cantidad de titulares', showgrid=True,
               gridcolor='#E0E0E0', gridwidth=1, griddash='dot'),
    margin=dict(t=60, b=50, l=60, r=40),
    width=750, height=480,
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.show()

## 4. Capa 2 — Candidatos para revisión

Se generan pares de nombres parecidos (fuzzy) por encima de un umbral alto. **Nada se fusiona acá.** Cada par se exporta a un CSV con la columna `misma_entidad` vacía, que se completa a mano con `si` / `no`.

Columnas de ayuda para decidir:
- `tipo_a` / `tipo_b`: si difieren (persona vs empresa), casi seguro son entidades distintas.
- `solo_en_a` / `solo_en_b`: los tokens que difieren entre ambos nombres.

In [55]:
#@title
# Generación de candidatos (fuzzy, no fusiona)

UMBRAL = 90
nombres = sorted(titulares['titular_norm'].tolist())
tipo_map = dict(zip(titulares['titular_norm'], titulares['tipo_actor']))

pares = []
for i, n in enumerate(nombres):
    for cand, score, _ in process.extract(n, nombres[i + 1:],
                                           scorer=fuzz.token_sort_ratio,
                                           score_cutoff=UMBRAL, limit=5):
        ta = set(n.replace(',', ' ').split())
        tb = set(cand.replace(',', ' ').split())
        pares.append({
            'nombre_a': n, 'nombre_b': cand, 'score': round(score, 1),
            'tipo_a': tipo_map[n], 'tipo_b': tipo_map[cand],
            'solo_en_a': ' '.join(sorted(ta - tb)),
            'solo_en_b': ' '.join(sorted(tb - ta)),
            'misma_entidad': ''
        })

candidatos = pd.DataFrame(pares).sort_values('score', ascending=False).reset_index(drop=True)
print(f"candidatos generados (score >= {UMBRAL}): {len(candidatos)}")
display(candidatos)

ruta_cand = MODIFIED / "titulares_candidatos.csv"
candidatos.to_csv(ruta_cand, index=False, encoding='utf-8-sig')
print("exportado:", ruta_cand)

candidatos generados (score >= 90): 7


,nombre_a,nombre_b,score,tipo_a,tipo_b,solo_en_a,solo_en_b,misma_entidad
0,DIEGO RUBEN OMAR,RUBEN OMAR DIEGO,100.0,persona_fisica,persona_fisica,,,
1,FELIX IGNACIO ROJAS,ROJAS FELIX IGNACIO,100.0,persona_fisica,persona_fisica,,,
2,HANCHA SA,HANICHA SA,94.7,empresa,empresa,HANCHA,HANICHA,
3,HOYOS SIMON AGUSTIN,SIMON AGUSTIN HOYOS SA,92.7,persona_fisica,empresa,,SA,
4,VALDEZ RENE ELADIO,VALDEZ RUBEN ELADIO,91.9,persona_fisica,persona_fisica,RENE,RUBEN,
5,"ARAUJO ADRIAN NICOLAS,ARAUJO JUAN CARLOS,ARAUJ...","ARAUJO ADRIAN NICOLAS,ARAUJO PABLO DANIEL,CANI...",90.5,persona_fisica,persona_fisica,CARLOS JUAN,,
6,RENE ELADIO VALDEZ SRL,VALDEZ RENE ELADIO,90.0,empresa,persona_fisica,SRL,,


exportado: catastro-minero-salta-mining-registry-salta/data/modified/titulares_candidatos.csv


In [56]:
#@title
# Verificación del guardrail: empresas de nombre parecido NO se marcan como iguales

familia = candidatos[candidatos['nombre_a'].str.contains('LITHIUM') |
                     candidatos['nombre_b'].str.contains('LITHIUM')]
print("pares candidatos entre empresas con 'LITHIUM':", len(familia))
if len(familia) == 0:
    print("Ninguna empresa LITHIUM distinta fue propuesta para fusión.")
else:
    display(familia)

pares candidatos entre empresas con 'LITHIUM': 0
Ninguna empresa LITHIUM distinta fue propuesta para fusión.


## 5. Capa 3 — Construcción del crosswalk auditable

Después de revisar los candidatos, se guarda el archivo como `titulares_candidatos_revisado.csv` en `data/modified/` con la columna `misma_entidad` completada. Esta sección solo fusiona los pares marcados con `si`.

Si el archivo revisado todavía no existe, el crosswalk resulta en identidad (cada titular es su propio canónico) y no se fusiona nada.

In [57]:
#@title
# Construcción del crosswalk (solo fusiona lo aprobado)

ruta_rev = MODIFIED / "titulares_candidatos_revisado.csv"

if ruta_rev.exists():
    rev = pd.read_csv(ruta_rev, encoding='utf-8-sig')
    aprobados = rev[rev['misma_entidad'].astype(str).str.lower().isin(['si', 'sí'])]
    print(f"pares aprobados para fusión: {len(aprobados)}")
else:
    aprobados = pd.DataFrame(columns=['nombre_a', 'nombre_b'])
    print("No se encontró titulares_candidatos_revisado.csv — crosswalk de identidad (sin fusiones).")

parent = {n: n for n in titulares['titular_norm']}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for _, r in aprobados.iterrows():
    if r['nombre_a'] in parent and r['nombre_b'] in parent:
        union(r['nombre_a'], r['nombre_b'])

freq = dict(zip(titulares['titular_norm'], titulares['n_registros']))
grupos = {}
for n in parent:
    grupos.setdefault(find(n), []).append(n)

filas = []
for miembros in grupos.values():
    canonico = sorted(miembros, key=lambda x: (freq.get(x, 0), len(x)), reverse=True)[0]
    for m in miembros:
        filas.append({'titular_norm': m, 'titular_canonico': canonico})

crosswalk = pd.DataFrame(filas)
ruta_cw = MODIFIED / "titulares_crosswalk.csv"
crosswalk.to_csv(ruta_cw, index=False, encoding='utf-8-sig')
print("exportado:", ruta_cw)

No se encontró titulares_candidatos_revisado.csv — crosswalk de identidad (sin fusiones).
exportado: catastro-minero-salta-mining-registry-salta/data/modified/titulares_crosswalk.csv


In [58]:
#@title
# Auditoría: qué se fusiona en qué

fusionados = {k: v for k, v in grupos.items() if len(v) > 1}
print(f"titulares antes: {len(titulares)}  ->  después del crosswalk: {crosswalk['titular_canonico'].nunique()}")
print(f"grupos fusionados: {len(fusionados)}\n")

for miembros in fusionados.values():
    canonico = crosswalk[crosswalk['titular_norm'] == miembros[0]]['titular_canonico'].iloc[0]
    print(f"CANÓNICO: {canonico}")
    for m in miembros:
        if m != canonico:
            print(f"   <- {m}")
    print()

titulares antes: 772  ->  después del crosswalk: 772
grupos fusionados: 0



In [59]:
#@title
# Exportar clasificación de actores

mapa_canonico = dict(zip(crosswalk['titular_norm'], crosswalk['titular_canonico']))

clasificacion = titulares[['titular_norm', 'tipo_actor', 'n_registros',
                           'superficie_ha', 'ejemplo_original']].copy()
clasificacion['titular_canonico'] = (clasificacion['titular_norm']
                                     .map(mapa_canonico).fillna(clasificacion['titular_norm']))
clasificacion = clasificacion[['titular_canonico', 'titular_norm', 'tipo_actor',
                               'n_registros', 'superficie_ha', 'ejemplo_original']]
clasificacion = clasificacion.sort_values(['tipo_actor', 'n_registros'],
                                          ascending=[True, False])

display(clasificacion)

ruta_clasif = MODIFIED / "clasificacion_actores.csv"
clasificacion.to_csv(ruta_clasif, index=False, encoding='utf-8-sig')
print("exportado:", ruta_clasif)

,titular_canonico,titular_norm,tipo_actor,n_registros,superficie_ha,ejemplo_original
197,COMUNIDAD TOBANTIRENDA,COMUNIDAD TOBANTIRENDA,comunidad,3,0.5190,COMUNIDAD TOBANTIRENDA
204,COOPERATIVA LA MESA REDONDA LTDA,COOPERATIVA LA MESA REDONDA LTDA,cooperativa,18,291.4002,COOPERATIVA LA MESA REDONDA LTDA.
203,COOPERATIVA DE TRABAJO MINACLAR LTDA,COOPERATIVA DE TRABAJO MINACLAR LTDA,cooperativa,6,227.1390,COOPERATIVA DE TRABAJO MINACLAR LTDA.
417,LITHIUM S CORP SA,LITHIUM S CORP SA,empresa,675,357940.5750,LITHIUM S CORPORATION S.A.
641,RINCON MINING LTD,RINCON MINING LTD,empresa,354,249009.6123,RINCON MINING LIMITED
...,...,...,...,...,...,...
747,VILCA GUTIERREZ ALVARO TOMAS,VILCA GUTIERREZ ALVARO TOMAS,persona_fisica,1,2997.1765,VILCA GUTIERREZ ALVARO TOMAS
748,VILCA GUTIERREZ MAGALI VITORIA,VILCA GUTIERREZ MAGALI VITORIA,persona_fisica,1,2997.1765,VILCA GUTIERREZ MAGALI VITORIA
752,"VIRGILI SAN MILLAN FERNANDO,VIRGILI SAN MILLAN...","VIRGILI SAN MILLAN FERNANDO,VIRGILI SAN MILLAN...",persona_fisica,1,3426.8000,"VIRGILI SAN MILLAN FERNANDO,VIRGILI SAN MILLAN..."
762,YAPURA MATIAS HUGO ALFREDO,YAPURA MATIAS HUGO ALFREDO,persona_fisica,1,1781.5646,YAPURA MATIAS HUGO ALFREDO


exportado: catastro-minero-salta-mining-registry-salta/data/modified/clasificacion_actores.csv


## 6. Aplicación en la limpieza

Para usar la titularidad normalizada en el resto de los notebooks, se agrega esta función a la carga: mapea `concesiona` a `titular_canonico` (o a `titular_norm` si no hay fusión).

In [60]:
#@title
# Función lista para consumir el crosswalk en la limpieza

mapa_canonico = dict(zip(crosswalk['titular_norm'], crosswalk['titular_canonico']))

def aplicar_normalizacion(g):
    g = g.copy()
    g['titular_norm'] = g['concesiona'].map(normalizar_titular)
    g['titular_canonico'] = g['titular_norm'].map(mapa_canonico).fillna(g['titular_norm'])
    g['tipo_actor'] = g['titular_norm'].map(tipo_actor)
    return g

In [61]:
# Ejecutá esta celda AL FINAL del notebook
import time
time.sleep(3)  # le da tiempo a Colab de sincronizar